# 01. Data Cleaning

**Scope**: This notebook covers only the four essential cleaning steps:
1. **Date filtering** — restrict to 2013–2016 issue years (mature, resolved outcomes)
2. **Data type conversion** — parse percentage strings, extract numeric term, etc.
3. **Missing value handling** — drop or impute depending on the variable
4. **Outlier handling** — cap extreme values at sensible bounds

Feature engineering and treatment variable construction are handled in **Notebook 02** after exploratory analysis.

In [1]:
import os
import time
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

os.chdir('C:/Users/David Cho/OneDrive/Desktop/Projects/All Lending Club Loan')

# Part A. Accepted Loan Data

In [3]:
DATA_DIR = './data'

ACCEPTED_COLS = [
    'id', 'loan_amnt', 'funded_amnt', 'term', 'int_rate',
    'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'annual_inc', 'verification_status', 'issue_d', 'loan_status',
    'purpose', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'application_type', 'addr_state'
]

start = time.time()
acc = pd.read_csv(
    f'{DATA_DIR}/accepted_2007_to_2018Q4.csv',
    usecols=ACCEPTED_COLS,
    low_memory=False
)
end = time.time()
print(f"Execution time: {end - start:.4f} seconds")
print(f'\nRaw shape : {acc.shape}')

Execution time: 12.4947 seconds

Raw shape : (2260701, 27)


## Step 1 — Date Filtering (2013–2016)

**Rationale**:
- 36-month loans issued through 2016 matured by early 2019, well before the 2018Q4 data cut-off → outcomes are largely resolved.
- 2017+ loans: many still labeled 'Current' → censored, downward-biased default rates.
- Pre-2013: smaller volume and structurally different market conditions.

In [11]:
acc['issue_d'] = pd.to_datetime(acc['issue_d'], format='%b-%Y')
acc['issue_year'] = acc['issue_d'].dt.year.astype('Int64')

print('Loan count by issue year (all):')
print(acc['issue_year'].value_counts().sort_index())

acc = acc[acc['issue_year'].between(2013, 2016)].copy()
print(f'\nAfter 2013-2016 filter: {acc.shape}')

Loan count by issue year (all):
issue_year
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134814
2014    235629
2015    421095
2016    434407
2017    443579
2018    495242
Name: count, dtype: Int64

After 2013-2016 filter: (1225945, 28)


In [13]:
# Keep a copy that retains 'Current' loans for the robustness check in Notebook 05.
# The main analysis uses only completed loans (known outcome).
acc_with_current = acc.copy()

COMPLETED_STATUSES = [
    'Charged Off',
    'Default',
    'Fully Paid',
    'Does not meet the credit policy. Status:Charged Off',
    'Does not meet the credit policy. Status:Fully Paid'
]

before = len(acc)
acc = acc[acc['loan_status'].isin(COMPLETED_STATUSES)].copy()
print(f'Completed loans: {len(acc):,}  (dropped {before - len(acc):,} Current / Late / Grace Period)')
print(acc['loan_status'].value_counts())

Completed loans: 1,026,558  (dropped 199,387 Current / Late / Grace Period)
loan_status
Fully Paid     820316
Charged Off    206230
Default            12
Name: count, dtype: int64


## Step 2 — Data Type Conversion

In [15]:
# --- Term: '36 months' → 36 (int) ---
acc['term_months'] = (
    acc['term'].str.strip().str.extract(r'(\d+)')[0].astype(int)
)

# --- Employment length: ordinal string → integer ---
EMP_MAP = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
acc['emp_length_num'] = acc['emp_length'].map(EMP_MAP)

# --- FICO mid-point: average of the reported low/high range ---
acc['fico_mid'] = (acc['fico_range_low'] + acc['fico_range_high']) / 2

## Step 3 — Missing Value Handling

In [17]:
print('Missing value counts (if any):')

missing = acc.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

Missing value counts (if any):
mths_since_last_delinq    506129
emp_length                 58454
emp_length_num             58454
revol_util                   535
dti                           40
inq_last_6mths                 1
dtype: int64


In [18]:
# --- mths_since_last_delinq: NaN means the borrower has NEVER been delinquent.
#     Fill with 999 as a sentinel value; also create a binary 'ever_delinq' flag.
acc['mths_since_last_delinq'] = acc['mths_since_last_delinq'].fillna(999)
acc['ever_delinq'] = (acc['mths_since_last_delinq'] < 999).astype(int)

# --- emp_length_num: NaN typically reflects 'n/a' (e.g., self-employed or unemployed).
#     Fill with -1 to distinguish from '< 1 year' (0).
acc['emp_length'] = acc['emp_length'].fillna('Unknown')
acc['emp_length_num'] = acc['emp_length_num'].fillna(-1)

# --- Drop rows with any missing column as they are trivial ---
before = len(acc)
acc = acc.dropna(axis=0)
print(f'Rows dropped (any missing columns): {before - len(acc):,}')
print(f'Remaining: {len(acc):,}')

print('\nMissing value counts after handling (if any):')
missing2 = acc.isna().sum()
print(missing2[missing2 > 0].sort_values(ascending=False))

Rows dropped (any missing columns): 576
Remaining: 1,025,982

Missing value counts after handling (if any):
Series([], dtype: int64)


## Step 4 — Outlier Handling

In [20]:
# annual_inc: a small number of entries exceed $10M — clearly data-entry errors.
# Cap at the 99th percentile; keep the original column for reference.
p99_inc = acc['annual_inc'].quantile(0.99)
acc['annual_inc_cap'] = acc['annual_inc'].clip(upper=p99_inc)
print(f'annual_inc : raw max = {acc["annual_inc"].max():,.0f}  |  capped at p99 = {p99_inc:,.0f}')

# revol_util: a few entries exceed 100%; cap at 100%.
high_util = (acc['revol_util'] > 100).sum()
acc['revol_util'] = acc['revol_util'].clip(upper=100) 
print(f'revol_util : raw max = {acc["revol_util"].max():,.0f}        |  capped at 100%')

# dti: a few entries exceed 100%; cap at 100%
high_dti = (acc['dti'] > 100).sum()
acc['dti'] = acc['dti'].clip(upper=100)
print(f'dti        : raw max = {acc["dti"].max():,.0f}        |  capped at 100%')

annual_inc : raw max = 9,550,000  |  capped at p99 = 250,000
revol_util : raw max = 100        |  capped at 100%
dti        : raw max = 100        |  capped at 100%


## Save Cleaned Data

In [22]:
# Main analysis dataset: completed loans, 2013-2016, outliers handled
acc.to_parquet(f'{DATA_DIR}/accepted_cleaned.parquet', index=False)

# Dataset including 'Current' loans for robustness check (Notebook 05)
acc_with_current.to_parquet(f'{DATA_DIR}/accepted_with_current.parquet', index=False)

print('Saved:')
print(f'  accepted_cleaned.parquet      {acc.shape}')
print(f'  accepted_with_current.parquet {acc_with_current.shape}')

Saved:
  accepted_cleaned.parquet      (1025982, 33)
  accepted_with_current.parquet (1225945, 28)


# Part B. Rejected Loans Data

In [24]:
# Load and clean rejected loans (same date filter + type conversions)
start = time.time()
rej = pd.read_csv(f'{DATA_DIR}/rejected_2007_to_2018Q4.csv', low_memory=False)
end = time.time()
print(f"Execution time: {end - start:.4f} seconds")
print(f'Raw rejected shape: {rej.shape}')

Execution time: 17.4375 seconds
Raw rejected shape: (27648741, 9)


## Step 1 — Data Filtering

In [26]:
rej['Application Date'] = pd.to_datetime(rej['Application Date'], errors='coerce')
rej['issue_year'] = rej['Application Date'].dt.year
rej = rej[rej['issue_year'].between(2013, 2016)].copy()

## Step 2 — Data Type Conversion

In [28]:
rej['dti'] = pd.to_numeric(
    rej['Debt-To-Income Ratio'].astype(str).str.replace('%', '').str.strip(),
    errors='coerce'
)
rej['emp_length_num'] = rej['Employment Length'].map(EMP_MAP).fillna(-1)
rej['Employment Length'] = rej['Employment Length'].fillna('Unknown')
rej['loan_amnt'] = pd.to_numeric(rej['Amount Requested'], errors='coerce')
rej['fico_mid'] = pd.to_numeric(rej['Risk_Score'], errors='coerce')

## Save Cleaned Data

In [30]:
start = time.time()
rej.to_parquet(f'{DATA_DIR}/rejected_cleaned.parquet', index=False)
end = time.time()
print(f"Execution time: {end - start:.4f} seconds")
print(f'Saved: rejected_cleaned.parquet  {rej.shape}')

Execution time: 3.1318 seconds
Saved: rejected_cleaned.parquet  (10323895, 14)


In [31]:
print('=== Cleaning Summary ===')
print(f'Accepted (completed, 2013-2016): {acc.shape}')
print(f'Rejected (2013-2016)           : {rej.shape}')

=== Cleaning Summary ===
Accepted (completed, 2013-2016): (1025982, 33)
Rejected (2013-2016)           : (10323895, 14)
